# 09 · Ablación de variables sociodemográficas y territoriales

**Objetivo.** Determinar qué grupos de variables agregan señal real manteniendo fijo el algoritmo. Se evita confundir una mejora por features con una mejora por cambiar simultáneamente de modelo.

**Baseline de MAE.** Última participación de la misma familia electoral para sección estable o componente oficial A+B (`lag_same_type_stable_plus_official_lineage`).

**Modelo constante.** `HistGradientBoostingRegressor` residual, con los mismos hiperparámetros, contracción 0.25 y corrección nacional de los notebooks anteriores.

**Grupos.** Historia mínima; demografía electoral por sexo/edad; contexto relativo a entidad/nación; combinación demografía+contexto; ECEG; combinación completa.

**Regla temporal.** Los grupos sin ECEG se prueban en 2018, 2021 y 2024. ECEG 2020 sólo se prueba en el origen 2024 porque su publicación oficial es posterior a la elección 2021.

**Salidas.** Mart enriquecido, predicciones, métricas, registro de variables, decisión de promoción y reporte HTML.

## Configuración y entradas

El algoritmo y sus hiperparámetros permanecen congelados. La única dimensión experimental es el conjunto de variables.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display

from radar27.modeling.feature_enrichment import (
    build_enriched_feature_mart,
    run_feature_ablation,
    write_feature_ablation_decision,
)
from radar27.modeling.entity_cycle_adaptation import run_entity_cycle_adaptation


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs/feature_ablation.yaml"
        ).exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
with (PROJECT_ROOT / "configs/feature_ablation.yaml").open(encoding="utf-8") as stream:
    ablation_config = yaml.safe_load(stream)
required_inputs = [
    PROJECT_ROOT / "artifacts/tables/feature_mart_ab.parquet",
    PROJECT_ROOT / "artifacts/tables/eceg_2020_section_features.parquet",
    PROJECT_ROOT / "artifacts/tables/national_shift_estimates.csv",
    PROJECT_ROOT / "artifacts/tables/backtest_metrics.csv",
]
input_checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required_inputs}
display(pd.Series(input_checks, name="existe").to_frame())
display(pd.DataFrame(ablation_config["feature_groups"]).T)
assert all(input_checks.values())

## Contexto territorial previo al target

Se calculan promedios y dispersiones de la participación rezagada por entidad y nación, así como la diferencia de cada unidad frente a esos referentes. Sólo se usan columnas conocidas antes del target.

In [ ]:
ablation_mart = build_enriched_feature_mart(PROJECT_ROOT)
context_checks = {
    "unit_target_unique": not ablation_mart.duplicated(
        ["target_year", "evaluation_unit_id"]
    ).any(),
    "context_strictly_prior": bool(
        (ablation_mart.context_source_max_year < ablation_mart.target_year).all()
    ),
    "geography_type_available": bool(ablation_mart.geography_unit_type.notna().all()),
}
display(pd.Series(context_checks, name="aprobado").to_frame())
display(ablation_mart[[
    "target_year", "evaluation_unit_id", "entity_same_type_mean",
    "same_type_minus_entity_mean", "national_same_type_mean",
    "geography_unit_type",
]].head())
assert all(context_checks.values())

## Backtests de ablación

Cada fila cambia únicamente el conjunto de variables. Se reporta el MAE ponderado después de aplicar la misma corrección nacional usada por la proyección experimental.

In [ ]:
ablation_predictions, ablation_metrics, ablation_registry = run_feature_ablation(
    PROJECT_ROOT, ablation_mart
)
display(ablation_metrics[[
    "target_year", "feature_group", "availability",
    "baseline_weighted_mae", "identified_weighted_mae",
    "mae_change_vs_baseline", "weighted_bias",
]])

In [ ]:
temporal_checks = {
    "eceg_only_2024": set(
        ablation_metrics.loc[
            ablation_metrics.availability.str.startswith("eceg_"), "target_year"
        ]
    ) == {2024},
    "strict_groups_all_folds": all(
        set(group.target_year) == {2018, 2021, 2024}
        for _, group in ablation_metrics[
            ablation_metrics.availability.eq("strict_all_targets")
        ].groupby("feature_group")
    ),
    "training_years_prior": all(
        all(int(year) < int(row.target_year) for year in str(row.training_target_years).split("|"))
        for row in ablation_registry.itertuples(index=False)
    ),
}
display(pd.Series(temporal_checks, name="aprobado").to_frame())
assert all(temporal_checks.values())

## Decisión

La promoción estricta exige mejorar 2021 y no deteriorar los otros folds más de 0.5 pp frente al conjunto actual. ECEG exige al menos 0.1 pp de mejora en 2024 para justificar complejidad y evidencia de un solo fold.

In [ ]:
ablation_decision = write_feature_ablation_decision(
    PROJECT_ROOT, ablation_metrics
)
display(pd.Series(ablation_decision).to_frame("valor"))

entity_cycle_predictions, entity_cycle_metrics, entity_cycle_decision = (
    run_entity_cycle_adaptation(PROJECT_ROOT)
)
display(pd.Series(entity_cycle_decision).to_frame("valor"))

In [ ]:
strict_pivot = ablation_metrics[
    ablation_metrics.availability.eq("strict_all_targets")
].pivot(index="feature_group", columns="target_year", values="identified_weighted_mae")
strict_pivot = 100 * strict_pivot
display(strict_pivot.rename(columns=lambda year: f"MAE_{year}_pp"))

eceg_2024 = ablation_metrics[
    ablation_metrics.target_year.eq(2024)
].sort_values("identified_weighted_mae")
display(eceg_2024[["feature_group", "availability", "identified_weighted_mae"]])

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/feature_ablation.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Ablación de variables</h1>",
    "<p>Generado por notebooks/09_feature_ablation.ipynb.</p>",
    "<h2>Métricas</h2>", ablation_metrics.to_html(index=False),
    "<h2>Registro</h2>", ablation_registry.to_html(index=False),
    "<h2>Decisión</h2><pre>" + json.dumps(
        ablation_decision, ensure_ascii=False, indent=2
    ) + "</pre>",
    "<h2>Adaptación estatal</h2>", entity_cycle_metrics.to_html(index=False),
    "<pre>" + json.dumps(entity_cycle_decision, ensure_ascii=False, indent=2) + "</pre>",
]), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "artifacts/tables/feature_ablation_mart.parquet",
    PROJECT_ROOT / "artifacts/tables/feature_ablation_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/feature_ablation_metrics.csv",
    PROJECT_ROOT / "artifacts/tables/feature_ablation_registry.csv",
    PROJECT_ROOT / "artifacts/feature_ablation_decision.json",
    PROJECT_ROOT / "artifacts/tables/entity_cycle_adaptation_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/entity_cycle_adaptation_metrics.csv",
    PROJECT_ROOT / "artifacts/entity_cycle_adaptation_decision.json",
    report_path,
]
final_checks = {
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    **context_checks,
    **temporal_checks,
}
display(pd.Series(final_checks, name="aprobado").to_frame())
assert all(final_checks.values())

In [ ]:
display(Markdown(
    f"**Cierre.** Grupo estricto seleccionado: "
    f"`{ablation_decision['selected_feature_group_for_2027']}`. "
    f"ECEG promovido: `{ablation_decision['eceg_promoted']}`. "
    f"Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))